In [1]:
import networkx as nx
import osmnx as ox
import numpy as np
import geopandas as gpd
from shapely.geometry import LineString

In [2]:
def calc_dist(G, orig_coords, dest):
    dest_x = G.nodes[dest]["x"]
    dest_y = G.nodes[dest]["y"]
    return round(ox.distance.great_circle(orig_coords[1], orig_coords[0], dest_y, dest_x))

In [29]:
def get_simple_graph(G, source_coords, beta=-0.1):
    gdf_nodes, gdf_edges = ox.convert.graph_to_gdfs(G)
    gdf_edges = gdf_edges.sort_index()
    gdf_edges_simple = gpd.GeoDataFrame(columns=list(gdf_edges.columns) + ['u','v','key', 'weight'])
    gdf_edges_simple = gdf_edges_simple.set_index(['u','v','key']) 

    for u,v,_ in list(gdf_edges.index):
        # remove self-edges
        if(u==v):
            continue
    
        # keep only unique edges
        if((u,v,0) in list(gdf_edges_simple.index)):
            continue
    
        ddist = calc_dist(G, source_coords, u) - calc_dist(G, source_coords, v)
        ddist_rev = calc_dist(G, source_coords, v) - calc_dist(G, source_coords, u)
        
        edges = gdf_edges.loc[(u,v)]
        if(edges.shape[0] == 1):
            new_edge = edges.iloc[0].copy()
            new_edge['weight'] = np.exp(beta * ddist)
            gdf_edges_simple.loc[(u,v,0)] = new_edge 

            # add reverse edge
            new_edge['weight'] = np.exp(beta * ddist_rev)
            gdf_edges_simple.loc[(v,u,0)] = new_edge 
        else: 
            new_edge = edges.sort_values(by='length').iloc[-1].copy() # longest edge
            new_edge['weight'] = np.exp(beta * ddist)
            gdf_edges_simple.loc[(u,v,0)] = new_edge

            # add reverse edge
            new_edge['weight'] = np.exp(beta * ddist_rev)
            gdf_edges_simple.loc[(v,u,0)] = new_edge 
            
    return ox.convert.graph_from_gdfs(gdf_nodes, gdf_edges_simple, graph_attrs=G.graph)

In [30]:
def random_walk(G, W, coords, steps): # coords are (lon,lat)
    orig = ox.distance.nearest_nodes(G, X=coords[0], Y=coords[1])
    nodes = list(G.nodes)

    n=len(nodes)
    idx = nodes.index(orig)
    node = nodes[idx]
    
    p = np.zeros(n)
    p[idx] = 1.0
    
    path = [node]
    for _ in range(steps):
        idx = np.random.choice(n, p=W[:,idx])
        next_node = nodes[idx]
        if(next_node != path[-1]):
            path.append(next_node)
    
    return path

In [31]:
G = ox.graph.graph_from_place("Piedmont, California, USA", network_type="drive")
# fig, ax = ox.plot.plot_graph(G)

In [32]:
source_coords = (-122.245846, 37.828903) # (lon,lat)
M = get_simple_graph(G, source_coords)
# fig, ax = ox.plot.plot_graph(M)

In [33]:
H = ox.convert.to_undirected(M)
n=len(H.nodes)
d = [d for _, d in H.degree(weight='weight')]
D = np.diag(d)
A = nx.adjacency_matrix(H).toarray()
L = nx.laplacian_matrix(H, weight='weight').toarray()
W = 0.5*(np.eye(n) + A @ np.linalg.inv(D))
pi= d/(np.ones(n).T @ d) # stable distribution

In [34]:
path = random_walk(M, W, source_coords, 23)
path

[53075602,
 53064327,
 53017091,
 53064327,
 53017091,
 53064327,
 53017091,
 53064327,
 53017091,
 53064327]

In [35]:
gdf = ox.routing.route_to_gdf(M, path)

In [36]:
gdf.explore()